# 4. Data Augmentation: SMILES Enumeration

In this notebook you will apply **SMILES enumeration** to your split datasets
as a data augmentation step before fine-tuning.

**What is SMILES enumeration?**
A molecule can be written as a SMILES string in many different ways — all
chemically equivalent, but textually different.  Enumeration exploits this
by generating multiple valid SMILES representations of the same molecule,
effectively giving the model more training signal per molecule.

This technique was introduced by **Bjerrum (2017)** and improves generalisation
of chemical language models by reducing the model's dependence on the canonical
atom-ordering convention.

**Why does this matter for fine-tuning?**
The pre-trained model was already trained on randomized SMILES (one random
enumeration per molecule).  To remain consistent with the pre-training
distribution, fine-tuning data should also be randomized.  You can additionally
choose a larger `augmentation_multiple` to increase the effective size of your
fine-tuning dataset — this is a tunable hyperparameter.

**Which splits to augment?**
Apply the same augmentation to **all three splits** (train, val, and test)
with the same `augmentation_multiple`.  This ensures all splits live in the
same SMILES representation space as the model.

> **Important — reporting statistics:** When computing metrics on an augmented
> set (e.g. in notebook 06), always report n as the number of *original*
> molecules, not the number of augmented rows.  Augmented rows are
> representations of the same molecule, not independent data points.

---

**Notebook structure:**
1. Import libraries
2. Load your dataset (one split at a time)
3. Understand the `Enumeration` class
4. Choose `augmentation_multiple` and run augmentation
5. Inspect and save the results
6. Repeat for all splits

---
## Step 1 — Install & Import Libraries

We need:
- `rdkit` — the standard cheminformatics library for working with molecules and SMILES
- `pandas` — for handling tabular data (our dataset)
- `abc` — Python's built-in module for defining abstract base classes
- `random` — to set a seed so results are reproducible

In [ ]:
import os
import random  # For setting a random seed (reproducibility)
import pandas as pd  # For reading CSV files and storing results as tables
import numpy as np  # For numerical operations (e.g., shuffling lists)
import matplotlib.pyplot as plt

# RDKit: the chemistry toolkit
from rdkit import Chem  # Core module: parse and manipulate molecules

from evaluation import load_smiles

# Set the random seed so that results are the same every time you run this notebook
random.seed(42)

print("All libraries imported successfully!")

In [ ]:
# Set the base directory to the current working directory (where this notebook is located)
base_dir = os.path.dirname(os.path.abspath("__file__"))

---
## Step 2 — Load Molecules

Replace the path below with your own file, choose one of the splits you created in notebook 2.

**Supported file formats:**
- `.smi` / `.txt` — one SMILES per line (first token is the SMILES, the rest is ignored)
- `.csv` — must contain a column named `smiles` (case-insensitive)

In [ ]:
# Load your SMILES from file: pick one split for now
smiles_path = os.path.join(base_dir, "your_path_here", "train.smi")

smiles_list = load_smiles(smiles_path)

# Show the first 5 molecules so we can confirm the data loaded correctly
print(f"Loaded {len(smiles_list)} molecules.\n")
for smi in smiles_list[:5]:
    print(smi)

### Step 2b — Quick sanity check on the SMILES

Before augmenting, it's good practice to verify that all SMILES strings are valid — i.e., RDKit can parse them into actual molecules.  
Invalid SMILES will cause errors later, so we filter them out here.

In [ ]:
valid_smiles = []
invalid_smiles = []

for smi in smiles_list:
    # Chem.MolFromSmiles() returns None if the SMILES is invalid
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        valid_smiles.append(smi)  # Keep valid ones
    else:
        invalid_smiles.append(smi)  # Track invalid ones for inspection

print(f"Valid SMILES:   {len(valid_smiles)}")
print(f"Invalid SMILES: {len(invalid_smiles)}")

if invalid_smiles:
    print("\nInvalid entries (will be excluded):")
    for s in invalid_smiles:
        print(" ", s)

# Use only the valid SMILES going forward
smiles_list = valid_smiles

---
## Step 3 — The Augmentation Class

In [ ]:
# Function that generates a random SMILES string for a given input SMILES string


def randomize_smiles(smiles):
    """Perform a randomization of a SMILES string
    must be RDKit sanitizable"""
    m = Chem.MolFromSmiles(smiles)
    ans = list(range(m.GetNumAtoms()))
    np.random.shuffle(ans)
    nm = Chem.RenumberAtoms(m, ans)
    return Chem.MolToSmiles(nm, canonical=False, isomericSmiles=False)

In [ ]:
class Enumeration:
    """
    Augmentation by SMILES enumeration (Bjerrum, 2017).

    For each molecule, generates multiple valid-but-different SMILES strings
    by randomising the starting atom of the graph traversal.
    Each unique string becomes an additional training example.
    """

    def __init__(self, smiles_list, augmentation_multiple):

        # Store the original list of SMILES strings
        self.smiles_list = smiles_list

        # Convert every SMILES to its canonical form using RDKit.
        # Canonical SMILES is a unique, standardised representation of a molecule.
        # This ensures we are not accidentally treating the same molecule as different ones.
        self.canonical_smiles = [
            Chem.MolToSmiles(Chem.MolFromSmiles(smiles)) for smiles in self.smiles_list
        ]

        # How many total versions of each molecule we want after augmentation.
        # E.g. augmentation_multiple=10 → 1 original + 9 augmented copies
        self.augmentation_multiple = augmentation_multiple

    def perform_augmentation(self, n_try=50, *args):
        """
        Perform SMILES enumeration augmentation.

        Parameters
        ----------
        n_try : int
            Maximum number of attempts to generate a new unique SMILES
            for a single molecule before giving up. Default is 50. Due to the nature
            of molecules, some may have very few unique SMILES representations, and this
            prevents infinite loops.

        Returns
        -------
        aug_df : pd.DataFrame
            A DataFrame with columns: ['smiles', 'index', 'type']
            - 'smiles' : the SMILES string (original or augmented)
            - 'index'  : integer pointing back to the original molecule
            - 'type'   : 'original' or 'augmented'
        """

        # aug_data will collect rows for the final DataFrame
        aug_data = []

        # all_aug tracks every augmented SMILES generated so far (across all molecules)
        # to make sure we never produce the same string twice in the whole dataset
        all_aug = []

        # Iterate over each molecule in the input list
        for idx, smiles in enumerate(self.smiles_list):
            # Always include the original SMILES as the first entry for this molecule
            aug_data.append([smiles, idx, "original"])

            # counter  : how many augmented versions we have successfully generated
            # try_aug  : how many consecutive failed attempts we have made
            counter, try_aug = 0, 0

            # aug_smiles holds augmented strings for *this* molecule only
            aug_smiles = []

            # Keep generating until we have (augmentation_multiple - 1) new versions.
            # We subtract 1 because the original already counts as one.
            while counter < self.augmentation_multiple - 1:
                # Generate a randomised SMILES for this molecule
                aug_string = randomize_smiles(smiles)

                # Accept the new SMILES only if it is genuinely different:
                #   1. Not already in the original dataset
                #   2. Not already added for this molecule
                #   3. Not already used for any other molecule
                if (
                    aug_string not in self.smiles_list
                    and aug_string not in aug_smiles
                    and aug_string not in all_aug
                ):
                    # Record the augmented SMILES, linked to the original molecule's index
                    aug_data.append([aug_string, idx, "augmented"])
                    aug_smiles.append(aug_string)  # Track for this molecule
                    all_aug.append(aug_string)  # Track globally
                    counter += 1
                    try_aug = 0  # Reset failure counter after a success
                else:
                    # This attempt did not produce a usable new string
                    try_aug += 1
                    if try_aug > n_try:
                        # We have failed n_try times in a row: this molecule
                        # likely cannot be enumerated further (e.g. very small molecule)
                        print(f"Could not fully enumerate molecule: {smiles}")
                        break  # Move on to the next molecule

        # Convert the collected rows into a DataFrame
        aug_df = pd.DataFrame(aug_data, columns=["smiles", "index", "type"])
        return aug_df

---
## Step 4 — Choose `augmentation_multiple` and Run

**`augmentation_multiple`** controls the total number of SMILES representations
per molecule (1 original + N randomized copies).

| Value | Meaning |
|---|---|
| 1 | No augmentation — only canonical SMILES (not recommended for fine-tuning) |
| 2–5 | Light augmentation — good for small datasets or when computation is limited |
| 10–20 | Heavier augmentation — can improve generalisation on larger datasets |


Canonical SMILES strings follow specific deterministic rules to represent molecules. These rules define a starting point for traversing the molecular graph, which means the same molecule always generates the same SMILES string. Because of this specific patterns might appear in your dataset: if your dataset consists of many substituted benzene derivatives, they will be represented by similar sequences. A model might overfit on these canonical patters and perform worse on generalization.

To avoid biasing the pre-training distribution toward any specific chemical series, 
the pre-trained model was trained with a set consisting one copy for each molecule: either a randomized or canonical SMILES string.
Using `augmentation_multiple=2` will result in a set with 2 copies per molecule, after which one of these should be selected to achieve a set like the pre-training one.

For fine-tuning, you can use a higher value to give the model more representations of your target molecules. We advise you to only select the augmented SMILES strings, so the model won't overfit on canonical patterns in your fine-tuning data.

**There is no single correct value** — treat it as a hyperparameter and
experiment.  A value between 5 and 15 is a reasonable starting point.

In [ ]:
augmentation_multiple = 10  # Total SMILES per molecule (1 original + N augmented)

# Create an instance of the Enumeration class
augmenter = Enumeration(
    smiles_list=smiles_list,  # Your list of valid SMILES
    augmentation_multiple=augmentation_multiple,
)

print(
    f"Augmenter ready. Will generate up to {augmentation_multiple}x versions per molecule."
)

In [ ]:
# Run the augmentation!
# This may take a few seconds depending on your dataset size and augmentation_multiple.
df_augmented = augmenter.perform_augmentation()

print("Augmentation complete.")
print(f"Original molecules : {len(smiles_list)}")
print(f"Total rows (orig + aug): {len(df_augmented)}")

---
## Step 5 — Inspect the Results

Let's look at what the augmented dataset looks like and verify it makes sense.

In [ ]:
# Display the first 15 rows
# You should see the original molecule followed by its augmented versions,
# all sharing the same 'index' value.
df_augmented.head(15)

In [ ]:
# Count how many original vs augmented entries we have
print("Row counts by type:")
print(df_augmented["type"].value_counts())

# Count how many augmented versions each molecule actually received
# (some small molecules may have received fewer than augmentation_multiple - 1)
aug_counts = df_augmented[df_augmented["type"] == "augmented"].groupby("index").size()

# Plot the counts, so we can see the number of augmented SMILES strings per index
ax = aug_counts.plot.bar(figsize=(12,5))
ax.set_ylabel("Number of augmented SMILES strings")
ax.set_ylim([0, augmentation_multiple])
ax.set_xlabel("Index in SMILES list")
ax.set_xticklabels(ax.get_xticklabels(), fontsize=7, rotation=0)
plt.tight_layout()



In [ ]:
# Let's zoom in on a single molecule to see all its augmented versions
# Here we look at molecule with index 0 (the first molecule in your dataset)
example_index = 0

example_rows = df_augmented[df_augmented["index"] == example_index]
print(f"Molecule #{example_index}: {smiles_list[example_index]}")
print(f"Total representations generated: {len(example_rows)-1}")
print(example_rows[["smiles", "type"]])

In [ ]:
# We only want to save the augmented SMILES strings, not the canonical one
df_augmented = df_augmented[df_augmented["type"] == "augmented"]
print(f"Total number of representations in this split after augmentation: {len(df_augmented)}")

In [ ]:
# Your job: Change the name of your output file as needed
# and save the augmented dataset to a CSV file.
output_path = os.path.join(base_dir, "your_path_here")
os.makedirs(
    os.path.dirname(output_path), exist_ok=True
)  # Ensure the output directory exists

df_augmented.to_csv(output_path, index=False)

print(f"Saved augmented dataset to: {output_path}")

---
## Step 6 — Repeat for All Splits

You have now augmented one split.  Repeat steps 2–5 for **each split**
(train, val, and test) using the **same `augmentation_multiple`**.

Use the augmented train and val sets for fine-tuning (notebook 05).
Keep the augmented test set aside — it is used for evaluation in notebook 06,
but remember: report test set statistics using n = original molecule count,
not the augmented row count.

**Things to try:**
- Does a higher `augmentation_multiple` improve the quality of the generated
  molecules (as measured in notebook 06)?
- What happens if you augment the training set more aggressively than the
  validation set?  Is this a good idea?